In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Reshape, Input

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test  = np.reshape(x_test,  (len(x_test),  28, 28, 1))

input_img = Input(shape=(28, 28, 1))
x = Flatten()(input_img)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
encoded = Dense(2, activation='linear')(x)

input_enc = Input(shape=(2,))
d = Dense(64, activation='relu')(input_enc)
d = Dense(28*28, activation='sigmoid')(d)
decoded = Reshape((28, 28, 1))(d)

encoder = keras.Model(input_img, encoded, name="encoder")
decoder = keras.Model(input_enc, decoded, name="decoder")
autoencoder = keras.Model(input_img, decoder(encoder(input_img)), name="autoencoder")
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

autoencoder.fit(x_train, x_train,
                epochs=10,
                batch_size=64,
                shuffle=True)

h = encoder.predict(x_test)
a = plt.scatter(h[:, 0], h[:, 1])

img = decoder.predict(np.expand_dims([50, 250], axis=0))
plt.imshow(img.squeeze(), cmap='gray')

## Izah

In [ ]:
import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.reshape      → massivin formasını dəyişmək (28x28 → 28x28x1)
# - np.expand_dims  → massivə əlavə ox (ölçü) əlavə etmək: (2,) → (1, 2)


import matplotlib.pyplot as plt
# matplotlib — şəkilləri və qrafikləri göstərmək üçün.
# Bu kodda:
# - plt.scatter  → nöqtə buludu (səpələnmə qrafiki) çəkmək
# - plt.imshow   → massivi şəkil kimi göstərmək
#
# DİQQƏT: əvvəlki kodda os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' sətri var idi
# (TensorFlow xəbərdarlıqlarını gizlədirdi). Burada yoxdur, ona görə konsolda
# daha çox texniki mesaj görünə bilər. Kodun işləməsinə təsir etmir.


from tensorflow.keras.datasets import mnist
# MNIST — əl ilə yazılmış rəqəmlərin (0-dan 9-a qədər) data bazası.
#   - 60000 öyrətmə şəkli, 10000 test şəkli
#   - hər şəkil 28x28 piksel, boz tonlu, piksel qiyməti 0..255


from tensorflow import keras
# Keras-ın özü. Bu kodda keras.Model(...) üçün lazımdır.


from tensorflow.keras.layers import Dense, Flatten, Reshape, Input
# Dense   → tam bağlı qat
# Flatten → (28, 28, 1) → (784,)
# Reshape → (784,) → (28, 28, 1)
# Input   → modelin giriş formasını elan edir


# =====================================================================
# BU KOD NƏDİR? — 2 ÖLÇÜLÜ GİZLİ FƏZALI AVTOENKODER
# =====================================================================
#
# Əvvəlki kodla (49 ölçülü darboğaz) müqayisədə İKİ əsas yenilik var:
#
#   1) Darboğaz cəmi 2 ədəddir (49 yox!). Yəni hər rəqəm şəkli (784 piksel)
#      yalnız 2 ədədlə (x, y koordinatı) təsvir olunur.
#      Üstünlüyü: 2 ədədi müstəvidə NÖQTƏ kimi çəkmək olur → gizli fəzanı
#      GÖZLƏ GÖRMƏK mümkündür.
#
#   2) Encoder və decoder AYRI-AYRI modellər kimi qurulub.
#      Əvvəlki kodda yalnız bütöv autoencoder var idi və "ortadakı kodu"
#      tək başına çıxarmaq olmurdu. İndi:
#         encoder(şəkil)  → 2 ədəd
#         decoder(2 ədəd) → şəkil
#      Bu, əvvəl danışdığımız "latent fəzada homotopiya" üçün lazım olan
#      şərtdir.
#
#   şəkil (784) ──► [ENCODER] ──► 2 ədəd (x, y) ──► [DECODER] ──► şəkil (784)
#
# ƏVVƏLKİ KODLA MÜQAYİSƏ:
#
#                       Əvvəlki kod              Bu kod
# Darboğaz ölçüsü:      49                       2
# Darboğaz aktivasiya:  relu                     linear
# Sıxılma nisbəti:      784/49 = 16 dəfə         784/2 = 392 dəfə
# Ayrıca encoder:       YOXDUR                   VAR
# Ayrıca decoder:       YOXDUR                   VAR
# epochs:               20                       10
# batch_size:           100                      64
# Bərpa keyfiyyəti:     orta (bir az bulanıq)    pis (çox bulanıq)
# Gizli fəzanı görmək:  mümkün deyil             mümkündür (scatter)
#
# NİYƏ 2 ÖLÇÜ? Bu, keyfiyyət üçün yox, GÖRSƏNMƏ üçündür. 2 ədədə sıxanda
# rəqəmlərin bərpası pisləşir, amma əvəzində "hansı rəqəm gizli fəzanın
# harasında yerləşir" sualına baxa bilərik.


# ---------------------------------------------------------------------
# 1) DATANIN YÜKLƏNMƏSİ VƏ HAZIRLANMASI
# ---------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Dörd massiv qaytarır:
#   x_train → (60000, 28, 28), uint8 (0..255)
#   y_train → (60000,) rəqəmlər (bu kodda ÖYRƏTMƏDƏ istifadə olunmur)
#   x_test  → (10000, 28, 28)
#   y_test  → (10000,)  (bu kodda da istifadə olunmur, amma scatter-i
#                        rənglənmək üçün istifadə etmək olar, aşağıya baxın)

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255
# (Rusca şərh: "giriş datasının standartlaşdırılması".)
# Piksellər 0..255 → 0..1. Səbəb: neyron şəbəkələr kiçik ədədlərlə yaxşı
# öyrənir və decoder-in sigmoid çıxışı da 0..1 aralığındadır.
# Bölmədən sonra tip float64 olur (Keras öyrətmədə float32-yə çevirir).

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))
# Kanal ölçüsü əlavə edir:
#   (60000, 28, 28) → (60000, 28, 28, 1)
#   (10000, 28, 28) → (10000, 28, 28, 1)
# Son 1 → tək kanal (boz ton). Elementlərin sayı dəyişmir.


# ---------------------------------------------------------------------
# 2) MODELLƏRİN QURULMASI (Functional API)
# ---------------------------------------------------------------------
#
# Bu dəfə İKİ ayrı "qatlar zənciri" qurulur, sonra üçüncü model onları
# birləşdirir.
#
#   ENCODER:
#   Giriş       Flatten     Dense      Dense      KOD
#  (28,28,1) ─► (784,) ─► (128,) ─► (64,) ─► (2,)
#
#   DECODER:
#   Giriş      Dense      Dense       Reshape
#    (2,)  ─► (64,) ─► (784,) ─► (28,28,1)
#
#   AUTOENCODER = decoder(encoder(giriş))   → ikisi ardıcıl birləşdirilir.

# ----- ENCODER hissəsi -----

input_img = Input(shape=(28, 28, 1))
# Encoder-in girişi: forma (28, 28, 1). Batch ölçüsü yazılmır.
# Real data deyil, "buraya şəkil gələcək" yerini tutan simvolik obyektdir.

x = Flatten()(input_img)
# (28, 28, 1) → (784,). Dense qat yalnız 1 ölçülü vektorla işləyir.
# Parametr: yoxdur.
# DİQQƏT: bu x dəyişəni şəbəkənin cari nöqtəsidir, x_train ilə qarışdırmayın.

x = Dense(128, activation='relu')(x)
# 784 → 128, ReLU. Parametr: 784 × 128 + 128 = 100480

x = Dense(64, activation='relu')(x)
# 128 → 64, ReLU. Parametr: 128 × 64 + 64 = 8256

encoded = Dense(2, activation='linear')(x)
# DARBOĞAZ qatı: 64 → 2. Parametr: 64 × 2 + 2 = 130
# Hər şəkil BU qatda cəmi 2 ədədə çevrilir: (x koordinatı, y koordinatı).
#
# NİYƏ 'linear' (yəni aktivasiyasız)?
#   - ReLU mənfi ədədləri 0 edərdi. 2 ölçüdə bu çox ziyanlıdır: neyronlardan
#     biri 0 qalsa, bütün nöqtələr bir xətt üzərində toplanar və ya
#     koordinatın yarısı itər.
#   - Linear halda kodlar istənilən ədəd ola bilər (mənfi də, çox böyük də).
#     Ona görə scatter-də nöqtələr bütün müstəvi boyu yayıla bilir.
#   - Əvvəlki kodda ReLU idi; ona görə "ölü neyron" riskindən yazmışdıq.
#
# DİQQƏT: linear olduğu üçün kodların SAHƏSİ məhdud deyil. Nöqtələrin
# hansı aralıqda olacağını əvvəlcədən bilmirik (məsələn, 0..300 ola bilər,
# -5..5 də ola bilər). Bu, aşağıda [50, 250] məsələsi üçün vacibdir.

# ----- DECODER hissəsi -----

input_enc = Input(shape=(2,))
# Decoder-in ayrıca girişi: 2 ədədlik vektor. Bu giriş encoder-dən ASILI DEYİL,
# ona görə decoder-ə istənilən 2 ədədi birbaşa verə bilərik (aşağıda [50, 250]).

d = Dense(64, activation='relu')(input_enc)
# 2 → 64, ReLU. Parametr: 2 × 64 + 64 = 192
# Kiçik koddan yenidən genişlənməyə başlayır.

d = Dense(28 * 28, activation='sigmoid')(d)
# 64 → 784, sigmoid (çıxış 0..1 aralığında, piksel aralığı ilə eyni).
# Parametr: 64 × 784 + 784 = 50960

decoded = Reshape((28, 28, 1))(d)
# (784,) → (28, 28, 1). Parametr: yoxdur.

# ----- Modellərin yığılması -----

encoder = keras.Model(input_img, encoded, name="encoder")
# Encoder modeli: şəkil girir, 2 ədəd çıxır.
# Parametr sayı: 100480 + 8256 + 130 = 108866.
# İstifadə: encoder.predict(şəkillər) → forma (şəkil_sayı, 2).

decoder = keras.Model(input_enc, decoded, name="decoder")
# Decoder modeli: 2 ədəd girir, şəkil çıxır.
# Parametr sayı: 192 + 50960 = 51152.
# İstifadə: decoder.predict(kodlar) → forma (kod_sayı, 28, 28, 1).

autoencoder = keras.Model(input_img, decoder(encoder(input_img)), name="autoencoder")
# Bütöv model: şəkil → encoder → 2 ədəd → decoder → şəkil.
# decoder(encoder(input_img)) → modeli "qat kimi" çağırmaq. Keras buna icazə
# verir: bir model digər modelin içində bir qat kimi işləyir.
#
# ÇOX VACİB: bu üç model ÇƏKİLƏRİ PAYLAŞIR (kopyalamır). Yəni autoencoder-i
# öyrətdikdə encoder və decoder də öyrənmiş olur. Ayrıca onlara fit()
# çağırmağa ehtiyac yoxdur və olmaz.
# Ümumi parametr sayı: 108866 + 51152 = 160018.
# (Əvvəlki kodda 166081 idi; fərq darboğazın kiçilməsindən gəlir.)

autoencoder.compile(optimizer='adam', loss='mean_squared_error')
# Loss = orta kvadratik səhv: ortalama( (orijinal piksel - bərpa piksel)² ).
# Optimizer: Adam, defolt öyrənmə sürəti 0.001.
# metrics yoxdur, çünki accuracy burada mənasızdır.
#
# compile YALNIZ autoencoder üçün edilib. encoder və decoder ayrıca
# compile olunmayıb. Bu normaldır, çünki onlara fit() çağırmırıq,
# yalnız predict() çağırırıq (predict compile tələb etmir).


# ---------------------------------------------------------------------
# 3) ÖYRƏTMƏ
# ---------------------------------------------------------------------

autoencoder.fit(x_train, x_train,
                epochs=10,
                batch_size=64,
                shuffle=True)
# fit(X, Y) yerinə fit(x_train, x_train): giriş də, cavab da eyni şəkildir.
#
# epochs=10      → bütün 60000 şəkil 10 dəfə keçilir.
# batch_size=64  → hər addımda 64 şəkil. 60000 / 64 = 937.5 → hər epoch-da
#                  938 addım (sonuncu batch yarımçıqdır, 32 şəkil).
#                  Cəmi təxminən 10 × 938 = 9380 çəki yeniləməsi.
# shuffle=True   → hər epoch-da sıra qarışdırılır (fit-in defoltudur).
#
# NƏ GÖZLƏMƏK LAZIMDIR? Loss əvvəlki koddan YÜKSƏK qalacaq (təxminən
# 0.04 civarı, dəqiq rəqəm hər işlətmədə dəyişir). Səbəb: 2 ədəd 784 pikselin
# məlumatını saxlamağa çatmır. Rəqəmin ümumi formasını saxlayır, incə
# detalları itirir.


# ---------------------------------------------------------------------
# 4) GİZLİ FƏZANIN GÖRÜNTÜSÜ
# ---------------------------------------------------------------------

h = encoder.predict(x_test)
# Bütün 10000 test şəklini encoder-dən keçirir.
# h-in forması: (10000, 2).
#   h[:, 0] → bütün şəkillərin 1-ci koordinatı (x oxu)
#   h[:, 1] → bütün şəkillərin 2-ci koordinatı (y oxu)
# Yəni hər şəkil müstəvidə bir nöqtədir.
# Burada yalnız ENCODER işləyir, decoder çağırılmır.

a = plt.scatter(h[:, 0], h[:, 1])
# Hər test şəklini müstəvidə nöqtə kimi çəkir (10000 nöqtə).
# a → çəkilmiş nöqtə kolleksiyasına istinaddır; sonra istifadə edilmir.
# Faydalı olan şey bu nöqtə buludunun ŞƏKLİDİR:
#   - Eyni rəqəmin şəkilləri adətən bir-birinə yaxın düşür (məsələn, bütün
#     "1"-lər bir bölgədə, bütün "0"-lar başqa bölgədə).
#   - Oxşar rəqəmlər (məsələn 4 və 9) yaxın və qismən üst-üstə düşür.
#   - Hər rəqəmin sinfi öyrətmədə İSTİFADƏ OLUNMAYIB, model bu qruplaşmanı
#     etiketsiz özü öyrənib.
# Burada rəng yoxdur, ona görə nöqtələr eyni rəngdədir. Hansı rəqəmin
# harada olduğunu görmək üçün y_test ilə rəngləmək olar:
#   plt.scatter(h[:, 0], h[:, 1], c=y_test, cmap='tab10', s=2)
#   plt.colorbar()
#
# DİQQƏT: plt.show() yazılmayıb. Jupyter/notebook-da qrafik avtomatik
# görünür. Adi .py faylında isə pəncərə açılmaz, plt.show() lazımdır.


# ---------------------------------------------------------------------
# 5) DECODER-İ ƏL İLƏ SEÇİLMİŞ NÖQTƏ İLƏ SINAQDAN KEÇİRMƏK
# ---------------------------------------------------------------------

img = decoder.predict(np.expand_dims([50, 250], axis=0))
# Bu sətir kodun ən maraqlı hissəsidir. Addım-addım:
#
#   [50, 250]                        → 2 ədədlik siyahı: x=50, y=250.
#                                      Bu, gizli fəzada bizim seçdiyimiz nöqtədir.
#   np.expand_dims(..., axis=0)      → forma (2,) → (1, 2).
#                                      Keras həmişə batch (qrup) gözləyir:
#                                      "1 ədəd kod, hər biri 2 ədədlik".
#   decoder.predict(...)             → bu koddan şəkil qurur. Çıxış: (1, 28, 28, 1).
#
# Nəticə: modelə heç bir real şəkil verməmişik, sadəcə gizli fəzada bir
# nöqtə göstərmişik və decoder o nöqtəyə uyğun rəqəmi "təsəvvür edib" çəkir.
# Bu, GENERASİYA deməkdir (yeni şəkil yaratmaq).
#
# NİYƏ MƏHZ [50, 250]? Bu rəqəmlər təsadüfi deyil. Yuxarıdakı scatter-ə
# baxıb nöqtə buludunun içində bir yer seçmişik. Vacib qeydlər:
#   1) Bu rəqəmlər hər öyrətmədə DƏYİŞİR. Çəkilər təsadüfi başladığı üçün
#      hər işlətmədə nöqtə buludu başqa yerdə və başqa miqyasda olur.
#      Başqa işlətmədə [50, 250] buludun çox kənarında ola bilər.
#   2) Əgər nöqtə buludun İÇİNDƏ deyilsə, decoder heç vaxt belə kod
#      görmədiyi üçün nəticə səs-küylü, mənasız lək kimi çıxa bilər.
#      Linear aktivasiya olduğu üçün model bunu qadağan etmir.
#   3) Düzgün yanaşma: öncə scatter-ə baxıb, bulud içindən nöqtə seçmək.

plt.imshow(img.squeeze(), cmap='gray')
# Decoder-in yaratdığı şəkli göstərir.
# .squeeze(): (1, 28, 28, 1) → (28, 28). imshow 2 ölçülü massiv gözləyir.
# cmap='gray': boz ton.
#
# ⚠ ƏSAS TUZAQ: bu sətir scatter-dən ƏVVƏL plt.show() və ya plt.figure()
# olmadan yazılıb. Notebook-da eyni xanada iki qrafik çəkiləndə imshow
# scatter-in ÜSTÜNƏ (eyni oxlara) çəkilə bilər. Nəticədə nöqtə buludunun
# üzərinə kiçik 28x28 şəkil yapışar və oxlar qarışar.
# Düzəliş: aralarına əlavə edin:
#   plt.show()     # 1-ci qrafiki bitir
#   plt.figure()   # ya da yeni pəncərə aç
# Və ya ayrı xanalara bölün.


# =====================================================================
# ƏLAVƏ: LATENT FƏZADA HOMOTOPİYA (əvvəlki söhbətimizin davamı)
# =====================================================================
# İndi encoder və decoder ayrı olduğuna görə, "daha düzgün" keçidi
# edə bilərik. Əvvəlki kodda interpolyasiya piksel səviyyəsində idi.
# Bu dəfə kodlar arasında olacaq:
#
#   frm, to = x_test[y_test == 5][1:3]       # iki "5" şəkli (əvvəlki kimi)
#
#   z_frm = encoder.predict(frm[None])[0]    # frm-in 2 ədədlik kodu
#   z_to  = encoder.predict(to[None])[0]     # to-nun 2 ədədlik kodu
#
#   ts = np.linspace(0., 1., 10)             # 10 addım
#   z  = np.array([z_frm * (1 - t) + z_to * t for t in ts])   # forma (10, 2)
#
#   plot_digits(decoder.predict(z))          # plot_digits əvvəlki koddakı kimi
#
# frm[None] → (28,28,1) → (1,28,28,1) (batch ölçüsü). [0] → batch-dan çıxarmaq.
# Bu halda xətt gizli müstəvidə iki nöqtə arasında düz xətdir və decoder
# xəttin hər nöqtəsini şəklə çevirir. Piksel qarışdırmasından fərqli olaraq
# "iki üst-üstə düşmüş 5" yox, formanın tədricən dəyişməsi alınır.
# 2 ölçüdə bərpa keyfiyyəti aşağı olduğu üçün şəkillər bulanıq olacaq.